# Italy Social Mobility Triangle (OECD-style)

This notebook analyzes Italy's social mobility structure using the precomputed triangle time series and links it to recent INVALSI socioeconomic achievement gaps.

Conceptual mapping used here:
- **Barrier Inequality**: structural inequality pressure
- **Barrier NEET**: transition blockage via youth inactivity
- **Mobility Opportunity**: effective opportunity side of the system

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'Notebooks' else Path.cwd()
OECD_DIR = ROOT / 'neet_outputs' / 'oecd_triangle'

triangle_path = OECD_DIR / 'italy_oecd_social_mobility_triangle_timeseries.csv'
escs_path = OECD_DIR / 'italy_invalsi_escs_gap_2024_2025.csv'

print('Using:', triangle_path)
print('Using:', escs_path)

In [ ]:
triangle = pd.read_csv(triangle_path)
escs = pd.read_csv(escs_path)

triangle = triangle.sort_values('year').reset_index(drop=True)
escs['escs_wle_gap_q4_q1'] = pd.to_numeric(escs['escs_wle_gap_q4_q1'], errors='coerce')

display(triangle)
display(escs.head(20))

## 1) Triangle Dynamics Over Time

In [ ]:
melted = triangle.melt(
    id_vars=['year'],
    value_vars=['share_inequality', 'share_neet', 'share_opportunity'],
    var_name='component',
    value_name='share'
)

name_map = {
    'share_inequality': 'Inequality Barrier',
    'share_neet': 'NEET Barrier',
    'share_opportunity': 'Opportunity'
}
melted['component'] = melted['component'].map(name_map)

plt.figure(figsize=(9, 5))
sns.lineplot(data=melted, x='year', y='share', hue='component', marker='o')
plt.title('Italy Mobility Triangle Shares Over Time')
plt.ylabel('Share')
plt.xlabel('Year')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
tri_rank = triangle[['year', 'share_inequality', 'share_neet', 'share_opportunity']].copy()
tri_rank['dominant_component'] = tri_rank[['share_inequality', 'share_neet', 'share_opportunity']].idxmax(axis=1)
tri_rank['dominant_component'] = tri_rank['dominant_component'].map(name_map)

tri_rank['imbalance'] = tri_rank[['share_inequality', 'share_neet', 'share_opportunity']].max(axis=1) - tri_rank[['share_inequality', 'share_neet', 'share_opportunity']].min(axis=1)
tri_rank = tri_rank.sort_values('year')
display(tri_rank)

## 2) Ternary Geometry (Cartesian Projection Provided in Data)
The triangle file already includes cartesian coordinates (`x`, `y`) derived from barycentric shares.

In [ ]:
plt.figure(figsize=(7, 6))

# Triangle vertices for reference
verts = np.array([[0, 0], [1, 0], [0.5, np.sqrt(3)/2], [0, 0]])
plt.plot(verts[:, 0], verts[:, 1], color='black', linewidth=1.5)

# Trajectory
plt.plot(triangle['x'], triangle['y'], marker='o', linewidth=2)
for _, r in triangle.iterrows():
    plt.text(r['x'] + 0.01, r['y'] + 0.01, str(int(r['year'])), fontsize=9)

plt.text(-0.02, -0.04, 'Inequality Barrier', fontsize=10)
plt.text(0.88, -0.04, 'NEET Barrier', fontsize=10)
plt.text(0.38, np.sqrt(3)/2 + 0.03, 'Opportunity', fontsize=10)

plt.title('Italy Mobility Position in OECD-style Triangle')
plt.xlim(-0.1, 1.1)
plt.ylim(-0.1, 1.0)
plt.axis('off')
plt.tight_layout()
plt.show()

## 3) Link with INVALSI ESCS Gaps (2024-2025)

In [ ]:
escs_valid = escs.dropna(subset=['escs_wle_gap_q4_q1']).copy()
escs_valid = escs_valid.sort_values('escs_wle_gap_q4_q1', ascending=False)
display(escs_valid[['GRADO', 'MATERIA', 'escs_wle_gap_q4_q1', 'school_year']])

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=escs_valid, x='escs_wle_gap_q4_q1', y='GRADO', hue='MATERIA')
plt.title('INVALSI ESCS Achievement Gap (Q4-Q1 WLE), 2024-2025')
plt.xlabel('Gap (higher = stronger socioeconomic gradient)')
plt.ylabel('Grade / Track')
plt.tight_layout()
plt.show()

In [ ]:
latest = triangle.loc[triangle['year'] == triangle['year'].max()].squeeze()
mean_gap = escs_valid['escs_wle_gap_q4_q1'].mean()

summary = pd.DataFrame({
    'metric': [
        'Latest year',
        'Opportunity share',
        'NEET barrier share',
        'Inequality barrier share',
        'Mean ESCS gap (INVALSI 2024-2025)'
    ],
    'value': [
        int(latest['year']),
        latest['share_opportunity'],
        latest['share_neet'],
        latest['share_inequality'],
        mean_gap
    ]
})
display(summary)

## 4) Interpretation Notes

Use this notebook as a policy diagnostic layer:
- If **NEET barrier share** grows, transitions into study/work are worsening.
- If **inequality barrier share** grows, background effects strengthen.
- If **opportunity share** rises while ESCS gaps narrow, mobility conditions are improving.

This is a compact monitoring architecture for Italy's social mobility path using your existing OECD-triangle outputs.